(class2-ex-prepare)=
# Exercise 1. Prepare Big (Text) Data for ML!
In this exercise, we will explore the dataset and perform the pre-processing needed for machine learning, regardless of whether it is text or not. 

After this, we'll be ready to vectorize our text with `bag-of-words` or `TF-IDF`.

## 1.1 Introducing the Data

As mentioned, you will be working with {cite:t}`dugan-etal-2024-raid`'s RAID dataset. The files can be found in `resources/data/raid` on `UCloud` (should be mounted if you followed [Class Setup](class-setup)):
```bash
└── raid
    ├── test.csv
    ├── test_none.csv <-- NEVER EVALUATE ON TEST BEFORE BEING DONE WITH TRAIN!!
    ├── train.csv
    └── train_none.csv
```

You will be working with the `train_none.csv` which contains the following:
```{figure} ../figures/class2/raid_figure.png
---
name: raid-overview
---
Figure modified from {cite:t}`dugan-etal-2024-raid`.
```

`train_none.csv` is a subset of the full dataset which also contains `adversarial attacks` (6.2M examples in total!). To keep it simple, which we won't be looking at these today.

:::{admonition} What are adversarial attacks?
:class: dropdown, tip
Adversarial attacks are input modifications (in our case, text) that exploit weaknesses the model did not encounter during training, causing incorrect predictions. Training with such attacks included can improve the classifier’s robustness against unexpected or manipulated inputs.

In {cite:t}`dugan-etal-2024-raid`'s RAID, examples include British spelling, article deletions, and mispellings. See the paper for details.

The full dataset `train.csv` is downloaded to `resources` for you to explore on your own (if you want to). If it loads slowly, consider a larger UCloud machine.
:::

### Load the Data
Start by importing `pathlib` and `pandas`:

In [1]:
from pathlib import Path
import pandas as pd

Define paths and load data:

In [2]:
# path of notebook
path = Path.cwd()

data_path = path.parents[1] / "resources" / "data" / "raid" / "train_none.csv"

In [3]:
raw_df = pd.read_csv(data_path)

Let's see how big our dataset is (number of rows):

In [4]:
print(len(raw_df))

467985


### Your Turn: Look at the Raw Data
```{admonition} HANDS-ON
:class: red

1. Load `raw_df` in your notebook if you haven't already! 
2. Print all column names  `raw_df` 
3. Do you notice any columns that you might not immediately know what corresponds to? Read up on [the column names](https://huggingface.co/datasets/liamdugan/raid#data-fields) before proceeding!
4. From {numref}`raid-overview`, we have gotten an overview of the kinds of LLMs used in this dataset, but what are they called in our dataframe? Find all unique values in the `models` column.
5. Finally, print a few example texts to look at!
```

#### Print Column Names

```{admonition} HINT
:class: tip, dropdown
Look at the .columns attribute on Pandas - see [docs](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.columns.html) for help
```

Check the solution:

In [5]:
columns_in_df = raw_df.columns.tolist() # you don't technically need .tolist() - it it just to get it in a neat list (try to remove to see effect) 
print(columns_in_df)

['id', 'adv_source_id', 'source_id', 'model', 'decoding', 'repetition_penalty', 'attack', 'domain', 'title', 'prompt', 'generation']


#### Unique Models

```{admonition} HINT
:class: tip, dropdown
What is the LLM column called? You need this, and then you can use the `.unique()` method:
https://pandas.pydata.org/docs/reference/api/pandas.unique.html
```

Solution below:

In [6]:
unique_models = raw_df["model"].unique()
print(unique_models)

['human' 'llama-chat' 'mpt' 'mpt-chat' 'gpt2' 'mistral' 'mistral-chat'
 'gpt3' 'cohere' 'chatgpt' 'gpt4' 'cohere-chat']


#### Print Example Texts
Solution below

In [7]:
print(f"### Generation 80 by {raw_df["model"][80]} ###")
print(raw_df["generation"][80])

### Generation 80 by human ###
We focus on an important yet challenging problem: using a 2D deep network to
deal with 3D segmentation for medical image analysis. Existing approaches
either applied multi-view planar (2D) networks or directly used volumetric (3D)
networks for this purpose, but both of them are not ideal: 2D networks cannot
capture 3D contexts effectively, and 3D networks are both memory-consuming and
less stable arguably due to the lack of pre-trained models.
  In this paper, we bridge the gap between 2D and 3D using a novel approach
named Elastic Boundary Projection (EBP). The key observation is that, although
the object is a 3D volume, what we really need in segmentation is to find its
boundary which is a 2D surface. Therefore, we place a number of pivot points in
the 3D space, and for each pivot, we determine its distance to the object
boundary along a dense set of directions. This creates an elastic shell around
each pivot which is initialized as a perfect sphere. We

In [8]:
print(f"### Generation 49000 by {raw_df["model"][459109]} ###")
print(raw_df["generation"][459109])

### Generation 49000 by cohere ###
 Bart Allen is the name of a fictional character appearing in American comic books published by DC Comics. The character was created by writer Len Wein and artist Mike Grell, and first appeared in Teen Titans vol. 1 #17 (November 1964).

Bart Allen is the nephew of Barry Allen, the Flash, and the grandson of Henry Allen and Nora Allen. He is also the son of Don Allen, the brother of Barry Allen. Bart Allen is the youngest person ever to gain the power of the Speed Force.

Before the name change to Bart Allen, the character was known as "Bartholomew Henry Allen III" and nicknamed "Tomboy". He is also nicknamed "Bart". When DC Comics changed the character's first name to "Bart", the character's age was changed from 12 to 15 years old. As "Bart Allen", the character became the fourth Flash. As a change to the character's backstory, Don Allen was retconned to be the brother of Barry Allen, instead of Henry Allen, Jr.; this made Bart Allen the nephew of th

## 1.2 Subset Data
We want to do a binary classification of `human` versus the LLM `cohere` which we also call our *classes* in ML. Let's subset the data to only include these, removing all other models. 

We will use the `isin()` function that we also played with in [Class 1 (Section 2.3) ](23-parts-of-speech-analysis):

In [9]:
df = raw_df[raw_df["model"].isin(["human", "cohere"])]

### Looking at the Classes

How many of each class do we have in our dataset? We can check this with the `group.by` function, applying `size()` to it:

In [10]:
df.groupby("model").size()

model
cohere    26742
human     13371
dtype: int64

:::{admonition} QUESTION
:class: red

Hmm, we seem to have double the amount of `cohere` generations as `human` texts. Do you know why that might be a problem?

Consider this with your group and click to reveal answer before proceeding.

```{dropdown} Click to see ANSWER
Classification models generally assume that all classes in a dataset have roughly the same number of examples. Unbalanced classes can cause the classifier to perform poorly on the under-represented class, also called the `minority class` (see {cite:t}`taskiran_comprehensive_2025`).
```
:::

As seen on {numref}`raid-overview`, we have more `cohere` rows because different generation parameters are used, producing two sets: greedy and sampling. We have ways of dealing with unbalanced classes, but we'll leave this for now!

```{admonition} LLM FRAMING: What is "greedy" and "sampling" ? 
:class: dropdown, fuchsia
In brief, `greedy` and `sampling` are *decoding* methods that determine how an LLM selects words when generating text. You will learn more about them later in the course!
```

### Creating a Numerical Label Variable
We want our classifier to predict `human` or `cohere`, but it doesn't really understand these labels. 

We'll add the label column `is_human` and assign `1` if the row `model` is `human` and `0` if it anything else (such as `cohere`):

In [11]:
df["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)

/var/folders/gg/gk923hkx2w3bw72pk2shplydry9j0b/T/ipykernel_42097/3636558662.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)


## 1.3 Create Training Splits!
When training a ML model, we need to consider three splits of the data:
```{figure} ../figures/class2/train_test.png
---
name: train-test-class2
---
Figure from [Rahul Chavan](https://medium.com/@rahulchavan4894/understanding-train-test-and-validation-dataset-split-in-simple-quick-terms-5a8630fe58c8)
```

Common percentage splits for train, val, and test are:
* 60%, 20%, 20% 
* 70%, 15%, 15%

### Install scikit-learn
`scikit-learn` is a widely used Python library for machine learning, covering supervised and unsupervised learning, model evaluation, and preprocessing. We will use it here for preprocessing.

Firstly, let's install `scikit-learn`: 

In [12]:
%pip install scikit-learn


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Now let's import the `train_test_split`:

In [13]:
from sklearn.model_selection import train_test_split

### Your Turn: Split Data with scikit-learn
Since `raid` has a seperate `test` set, we only need to split our data into `train` and `val`:

:::{admonition} HANDS-ON
:class: red
Instead of showing you the code this time, I'll ask you to check [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) and [this guide](https://medium.com/@whyamit404/understanding-train-test-split-in-pandas-eb1116576c66) to:

1. Use the function `train_test_split()` to split your `df` into `train_df` and `val_df`. The size of our validation set should be `20 %`

```{dropdown} Why check documentation?
This is how real coders do their work! While ChatGPT might help you with a code snippet, if you want to understand a function OR use it for a more specific case than the general example, the documentation is a great place to be!
```

*Remember we have test data in a seperate file! And we won't touch it today...*
:::

:::{admonition} Can you help me breakdown the function?
:class: tip, dropdown

Let's look at the docs together:
```{image} ../figures/class2/train_test_split.png
:alt: train_test_split_docs
:width: 500px
```
&nbsp;
- **`arrays`**: the data variables to be split. These can be:
  - A complete dataframe `df` (our case).
  - Pre-defined (NumPy) arrays `X` and `y`  
    - `X` = features to train on (e.g., our text) 
    - `y` = labels (to predict)
  - Pandas Series, e.g.  
    - `X = df_balanced["generation"]`  
    - `y = df_balanced["is_human"]`
  - A dataframe with multiple features, e.g.  
    - `X = df[["average_sentence_length", "mean_dependency_distance"]]`

- **`test_size` / `train_size`**: specify the proportion of data for testing or training.  
  - Only one needs to be set — the other will be inferred to make up 100%.

- **`random_state`**: ensures reproducibility of the split (same role as a `seed`), so you always get the same split.

- **`shuffle`**: defaults to `True`, which is usually desirable. No need to set it explicitly unless you want to disable shuffling.

- **`stratify`**: pass the class variable here to preserve the class distribution across splits.

---

**Examples**  
Split a dataframe directly (returns two dataframes):  
```python
train_df, val_df = train_test_split(df, ...)
```

Alternatively, if you insert `X` (features) as the first argument and `y` (labels) as the second, the `train_test_split()` **returns** four variables that you unpack as such:
```python
X_train, X_val, y_train, y_val = train_test_split(df["generation"], df["is_human"], ...)
```
:::

Solution if you are stuck or want to compare:

In [14]:
train_df, val_df= train_test_split(
                                                    df,
                                                    test_size=0.20,
                                                    random_state=42,
                                                    stratify=df["is_human"]
                                                    )